# attention language model

Start from the same Tiny Shakespeare token stream as the bigram model, then run explicit multi-head causal self-attention on the embedded batch.

In [ ]:
import random
import sys
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.dataset import get_batch, load_tiny_shakespeare_tokens, split_token_stream

## prepare token streams and batches

In [ ]:
tokens, vocab, _ = load_tiny_shakespeare_tokens(repo_root / "data")
train_tokens, validation_tokens = split_token_stream(tokens)

vocab_size = len(vocab)
block_size = 8
batch_size = 32
n_embd = 32
num_heads = 2
head_size = n_embd // num_heads

random.seed(42)
x_batch, y_batch = get_batch("train", train_tokens, validation_tokens, block_size, batch_size)
x_batch = torch.tensor(x_batch, dtype=torch.long)
y_batch = torch.tensor(y_batch, dtype=torch.long)

## embed the current batch

In [ ]:
token_embedding_table = nn.Embedding(vocab_size, n_embd)
x = token_embedding_table(x_batch)

assert x.shape == (batch_size, block_size, n_embd)
tuple(x.shape)

## multi-head causal self-attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, d_model, d_k, d_v):
        super().__init__()
        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v

        self.w_qs = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_ks = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_vs = nn.Linear(d_model, n_head * d_v, bias=False)
        self.output_projection = nn.Linear(n_head * d_v, d_model, bias=False)

    def forward(self, query, key, value):
        d_k, d_v, n_head = self.d_k, self.d_v, self.n_head
        B, len_q, _ = query.shape
        _, len_k, _ = key.shape
        _, len_v, _ = value.shape

        q = self.w_qs(query).view(B, len_q, n_head, d_k)
        k = self.w_ks(key).view(B, len_k, n_head, d_k)
        v = self.w_vs(value).view(B, len_v, n_head, d_v)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # scaled dot-product attention
        # q @ k^T: (B, h, T, d_k) @ (B, h, d_k, T)
        #          -> (B, h, T, T)
        scores = q @ k.transpose(-2, -1)
        scores = scores / (d_k**0.5)

        # causal mask
        causal_mask = torch.triu(
            torch.ones(len_q, len_k, device=scores.device, dtype=torch.bool), diagonal=1
        )
        scores = scores.masked_fill(causal_mask, float("-inf"))

        # normalize, then retrieve values
        weights = torch.softmax(scores, dim=-1)  # (B, h, T, T)
        output = weights @ v  # (B, h, T, d_v)

        # put heads beside each other again
        output = output.transpose(1, 2).contiguous()  # (B, T, h, d_v)
        output = output.view(B, len_q, n_head * d_v)  # (B, T, h*d_v)
        output = self.output_projection(output)  # (B, T, C)

        return output, weights

## position-wise feed-forward network

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.w_2(F.relu(self.w_1(x)))

## attention shape check

In [ ]:
x = torch.randn(batch_size, block_size, n_embd)
mha = MultiHeadAttention(n_head=num_heads, d_model=n_embd, d_k=head_size, d_v=head_size)
out, weights = mha(x, x, x)

assert out.shape == x.shape
assert weights.shape == (batch_size, num_heads, block_size, block_size)
tuple(out.shape)

## transformer block

Combine attention and feed-forward transformations with pre-normalization and residual connections.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, d_k, d_v, d_ff):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(n_head, d_model, d_k, d_v)

        self.ln2 = nn.LayerNorm(d_model)
        self.ff = PositionWiseFeedForward(d_model, d_ff)

    def forward(self, x):
        # attention residual
        z = self.ln1(x)
        attn_out, weights = self.attn(z, z, z)
        x = x + attn_out

        # feed-forward residual
        x = x + self.ff(self.ln2(x))

        return x, weights

## block shape check

In [ ]:
block = TransformerBlock(
    d_model=n_embd,
    n_head=num_heads,
    d_k=head_size,
    d_v=head_size,
    d_ff=4 * n_embd,
)

out, weights = block(x)

print(x.shape)
print(out.shape)
print(weights.shape)

## train a one-block language model

the shared next-token loss trains all registered model parameters together. this notebook uses BPE tokens, not individual characters. define the loop here; the implemented one-block wrapper is in `src/language_models.py`. plain minibatch SGD exposes the update rule directly; the optimizer and learning rate are experimental choices.


In [ ]:
def train_language_model(model, *, steps, learning_rate, eval_every=100, eval_batches=10):
    """Train a model mapping token ids [B, T] to raw vocabulary logits [B, T, V]."""
    if min(steps, eval_every, eval_batches) < 1 or learning_rate <= 0:
        raise ValueError("steps, evaluation counts, and learning rate must be positive")

    device = next(model.parameters()).device
    # plain minibatch SGD makes the update rule we discussed explicit.
    # construct once: every registered trainable component shares this objective.
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    history = []

    def batch_loss(split):
        inputs, targets = get_batch(split, train_tokens, validation_tokens, block_size, batch_size)
        inputs = torch.tensor(inputs, dtype=torch.long, device=device)
        targets = torch.tensor(targets, dtype=torch.long, device=device)
        logits = model(inputs)
        # one prediction per position; flatten B and T without mixing vocabulary scores.
        # cross_entropy takes raw logits, so do not apply softmax first.
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

    model.train()
    for step in range(1, steps + 1):
        optimizer.zero_grad(set_to_none=True)  # discard the previous batch's gradients
        loss = batch_loss("train")  # forward pass and shared next-token objective
        loss.backward()  # chain rule: calculate gradients throughout the model
        optimizer.step()  # SGD: change parameters using those gradients

        if step == 1 or step % eval_every == 0 or step == steps:
            model.eval()  # evaluation behavior; this alone does not disable gradients
            with torch.no_grad():
                validation_loss = (
                    sum(batch_loss("validation").item() for _ in range(eval_batches)) / eval_batches
                )
            model.train()
            # training loss is this update's batch loss, measured before the update.
            history.append(
                {"step": step, "train_batch_loss": loss.item(), "validation_loss": validation_loss}
            )
            print(f"step {step}: train batch {loss.item():.4f}, validation {validation_loss:.4f}")
    return history


# the implemented wrapper lives in src/language_models.py:
# token ids -> token + position embeddings -> TransformerBlock (unpack its pair)
#           -> final layer norm -> vocabulary projection -> raw logits.
# all components are registered so model.parameters() includes them.
# TODO: choose steps and learning_rate explicitly, then call train_language_model.
# rerunning the function continues the supplied model's current weights.
# TODO: inspect both losses; a lower training batch loss alone is not generalization.

## compare bigram and one-block outputs

evaluate both trained models on identical sampled training and validation windows, then generate from the same prompt. delta means transformer minus bigram cross-entropy (nats per BPE token); negative favors the transformer. this is a between-model gap, not a before/after training change.

record each training budget and optimizer when interpreting results: the bigram notebook uses AdamW, while the loop above uses SGD. these models also have different parameter counts; this comparison alone does not isolate architecture.


In [ ]:
def compare_language_models(
    bigram_model, transformer_model, *, prompt_ids, new_tokens=100, loss_batches=10, seed=42
):
    """Compare trained models using shared BPE ids and identical loss-evaluation batches."""
    from src.tokenizer import bpe_decode

    if not prompt_ids or new_tokens < 0 or loss_batches < 1:
        raise ValueError("provide a nonempty prompt and valid generation/evaluation counts")
    models = {"bigram": bigram_model, "single transformer": transformer_model}
    # use a local RNG so comparison does not change future training batch sampling.
    rng = random.Random(seed)
    batches = {}
    for split, stream in [("train", train_tokens), ("validation", validation_tokens)]:
        batches[split] = []
        for _ in range(loss_batches):
            starts = [rng.randrange(len(stream) - block_size) for _ in range(batch_size)]
            inputs = [stream[i : i + block_size] for i in starts]
            targets = [stream[i + 1 : i + block_size + 1] for i in starts]
            batches[split].append((inputs, targets))

    def get_logits(model, ids):
        result = model(ids)
        # the existing bigram returns (logits, loss); the new wrapper returns logits.
        return result[0] if isinstance(result, tuple) else result

    results = {}
    for name, model in models.items():
        device = next(model.parameters()).device
        was_training = model.training
        model.eval()
        try:
            with torch.no_grad():
                losses = {}
                for split, shared_batches in batches.items():
                    total = 0.0
                    for inputs, targets in shared_batches:
                        inputs = torch.tensor(inputs, dtype=torch.long, device=device)
                        targets = torch.tensor(targets, dtype=torch.long, device=device)
                        logits = get_logits(model, inputs)
                        total += F.cross_entropy(
                            logits.reshape(-1, logits.size(-1)), targets.reshape(-1)
                        ).item()
                    losses[split] = total / loss_batches

                ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)
                sampler = torch.Generator(device="cpu").manual_seed(seed)
                for _ in range(new_tokens):
                    # keep the transformer inside its context window; bigram uses the last token.
                    logits = get_logits(model, ids[:, -block_size:])[:, -1, :]
                    probabilities = torch.softmax(logits, dim=-1).cpu()
                    next_id = torch.multinomial(probabilities, 1, generator=sampler).to(device)
                    ids = torch.cat((ids, next_id), dim=1)
                results[name] = {
                    "train_loss": losses["train"],
                    "validation_loss": losses["validation"],
                    "text": bpe_decode(ids[0].tolist(), vocab, errors="replace"),
                    "parameters": sum(p.numel() for p in model.parameters()),
                }
        finally:
            model.train(was_training)

    print("same prompt, temperature 1, same sampling seed; one sample is qualitative evidence")
    for name, result in results.items():
        print(f"\n{name} ({result['parameters']:,} parameters):\n{result['text']}")
        print(
            f"train loss: {result['train_loss']:.4f}; validation loss: {result['validation_loss']:.4f}"
        )
    for metric in ("train_loss", "validation_loss"):
        delta = results["single transformer"][metric] - results["bigram"][metric]
        print(f"{metric} delta (transformer - bigram): {delta:+.4f} nats/token")
    return results


# the next cell loads both trained models and calls this comparison.

## paired control and treatment experiment

three paired seeds; identical batches and shared-component initialization; 10,000 uninterrupted AdamW updates each, rerun from the same initial seeds. the treatment adds learned positions, one transformer block, and final normalization. equal updates are not equal compute or parameter counts. validation loss is the primary outcome; these exploratory runs are not evidence of universal superiority. generated weights and results stay local under `artifacts/`.

only the training budget changed from the 1,000-update run. compare both budgets below: a widening, stable, or closing gap are all informative outcomes. full checkpoints include optimizer and random-generator state; the original short-run artifacts remain available.


In [ ]:
import json

from IPython.display import Image, display

from src.language_models import BigramLanguageModel, SingleTransformerLanguageModel

# reproduce all three paired runs from the repository root with:
# uv run python -m experiments.compare_bigram_transformer --steps 10000 \
#     --output artifacts/bigram-vs-transformer-10k
experiment_dir = repo_root / "artifacts" / "bigram-vs-transformer-10k"
report = json.loads((experiment_dir / "results.json").read_text())
print(report["config"])
# compare fixed evaluation windows at the two preselected training budgets.
for budget in (1000, 10000):
    for split in ("train", "validation"):
        means = {}
        for name in ("bigram", "transformer"):
            values = [
                h[split]
                for run in report["runs"]
                if run["model"] == name
                for h in run["history"]
                if h["step"] == budget
            ]
            means[name] = sum(values) / len(values)
        print(
            f"{budget:5d} updates, {split}: "
            f"bigram={means['bigram']:.4f}, transformer={means['transformer']:.4f}, "
            f"delta={means['transformer'] - means['bigram']:+.4f} nats/token"
        )

for seed in report["config"]["seeds"]:
    pair = {r["model"]: r for r in report["runs"] if r["seed"] == seed}
    for split in ("train", "validation"):
        control = pair["bigram"]["history"][-1][split]
        treatment = pair["transformer"]["history"][-1][split]
        print(
            f"seed {seed} {split}: bigram={control:.4f}, transformer={treatment:.4f}, "
            f"delta={treatment - control:+.4f} nats/token"
        )
display(Image(filename=str(experiment_dir / "loss.png")))

# inspect the first preselected seed, rather than selecting the best-looking run.
seed = report["config"]["seeds"][0]
bigram_model = BigramLanguageModel(vocab_size, n_embd)
single_transformer_model = SingleTransformerLanguageModel(vocab_size, n_embd, block_size, num_heads)
bigram_model.load_state_dict(torch.load(experiment_dir / f"bigram-{seed}.pt", weights_only=True))
single_transformer_model.load_state_dict(
    torch.load(experiment_dir / f"transformer-{seed}.pt", weights_only=True)
)
comparison = compare_language_models(
    bigram_model, single_transformer_model, prompt_ids=train_tokens[:8]
)

## text at 1k, 2.5k, 5k, and 10k updates

same prompt and sampling seed at each checkpoint, with 100 new BPE tokens. compare changes within each model as well as between models; the samples illustrate behavior, while the loss curves provide broader evidence.


In [ ]:
from experiments.plot_checkpoint_text import plot_checkpoint_text

# reproduce the preselected seed's checkpoints, without selecting attractive samples:
# from experiments.compare_bigram_transformer import run
# run(steps=10000, seeds=(42,), output=str(repo_root / "artifacts/bigram-vs-transformer-milestones"))
milestone_dir = repo_root / "artifacts" / "bigram-vs-transformer-milestones"
figure_path = plot_checkpoint_text(milestone_dir, seed=42)
display(Image(filename=str(figure_path)))

## compose transformer blocks

each block refines the preceding representation with separate parameters. depth and block dimensions are configurable. the stack returns final embeddings and a list of attention maps in layer order; vocabulary projection remains the language-model wrapper’s responsibility. the reusable version is in `src/language_models.py`.


In [ ]:
class TransformerStack(nn.Module):
    """Compose independent blocks; return embeddings and attention maps in layer order."""

    def __init__(self, n_layers, d_model, n_head, d_k, d_v, d_ff):
        super().__init__()
        if n_layers < 1:
            raise ValueError("n_layers must be positive")
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_head, d_k, d_v, d_ff) for _ in range(n_layers)]
        )

    def forward(self, x):
        # x stays [batch, tokens, d_model] throughout the stack.
        attention_weights = []
        for block in self.blocks:
            x, weights = block(x)
            attention_weights.append(weights)
        return x, attention_weights

In [ ]:
# use the same block objects to verify sequential composition.
stack = TransformerStack(
    n_layers=2,
    d_model=n_embd,
    n_head=num_heads,
    d_k=head_size,
    d_v=head_size,
    d_ff=4 * n_embd,
)
# dummy embeddings let us check the wiring without loading text.
stack_input = torch.randn(2, block_size, n_embd)
stack_output, layer_weights = stack(stack_input)
first_output, _ = stack.blocks[0](stack_input)
explicit_output, _ = stack.blocks[1](first_output)
torch.testing.assert_close(stack_output, explicit_output)
assert stack_output.shape == stack_input.shape
assert len(layer_weights) == 2
# compare object identities, not parameter values: each layer learns independently.
first_block_parameter_ids = {id(parameter) for parameter in stack.blocks[0].parameters()}
second_block_parameter_ids = {id(parameter) for parameter in stack.blocks[1].parameters()}
shared_parameter_ids = first_block_parameter_ids.intersection(second_block_parameter_ids)
assert len(shared_parameter_ids) == 0, "blocks must have separate parameters"
print("stack output:", tuple(stack_output.shape))
attention_shapes = [tuple(weights.shape) for weights in layer_weights]
print("attention per layer:", attention_shapes)
print("composition and independent-parameter checks passed")

## all treatments: matched 10k-update comparison

compare bigram, one transformer block, and two transformer blocks across seeds 42, 43, and 44. all models see the same batches and fixed evaluation windows. width 32, context 8, batch size 32; AdamW at 0.001 with weight decay 0.01. shared components start at identical values, with separate parameter objects. the second block adds parameters and compute.

question and pre-run hypothesis, controls, checkpoints, and reproduction command are in `experiments/depth-comparison.md`. validation loss is primary. the curves show the mean and full seed range, not confidence intervals. samples use the first preselected seed and are qualitative. full optimizer/RNG checkpoints and source hashes are saved locally.


In [ ]:
import json
from pathlib import Path
from statistics import mean

from IPython.display import Image, display

comparison_root = Path.cwd()
if not (comparison_root / "artifacts").exists():
    comparison_root = comparison_root.parent
all_treatments_directory = comparison_root / "artifacts" / "all-treatments-10k"
all_treatments_report = json.loads((all_treatments_directory / "results.json").read_text())
treatment_names = ["bigram", "transformer", "two_blocks"]

print("final cross-entropy: nats per BPE token; update time excludes evaluation")
for treatment_name in treatment_names:
    treatment_runs = [
        run for run in all_treatments_report["runs"] if run["model"] == treatment_name
    ]
    training_losses = [run["history"][-1]["train"] for run in treatment_runs]
    validation_losses = [run["history"][-1]["validation"] for run in treatment_runs]
    update_times = [run["update_seconds"] for run in treatment_runs]
    parameter_count = treatment_runs[0]["parameters"]
    print(
        f"{treatment_name}: parameters={parameter_count:,}, "
        f"train={mean(training_losses):.4f}, validation={mean(validation_losses):.4f}, "
        f"validation range=[{min(validation_losses):.4f}, {max(validation_losses):.4f}], "
        f"mean update seconds={mean(update_times):.1f}"
    )

print("\npaired validation losses and depth differences")
for seed in all_treatments_report["config"]["seeds"]:
    losses_by_treatment = {}
    for run in all_treatments_report["runs"]:
        if run["seed"] == seed:
            losses_by_treatment[run["model"]] = run["history"][-1]["validation"]
    depth_difference = losses_by_treatment["two_blocks"] - losses_by_treatment["transformer"]
    print(f"seed {seed}: {losses_by_treatment}; two minus one={depth_difference:+.4f}")

display(Image(filename=str(all_treatments_directory / "loss.png")))
print("\nseed 42 samples: same prompt, temperature 1, sampling seed 123")
for run in all_treatments_report["runs"]:
    if run["seed"] == 42:
        print(f"\n{run['model']}\n{run['sample']}")